In [3]:
import os
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns 
import keras
import tensorflow as tf 
from keras.preprocessing.image import load_img
from tensorflow.keras.preprocessing.image import ImageDataGenerator
tf.config.experimental_run_functions_eagerly(True)

Instructions for updating:
Use `tf.config.run_functions_eagerly` instead of the experimental version.


In [4]:
image_dir = 'D:/myvenv/archive/brain_tumor_dataset' # directory where dataset is present

In [5]:
BATCH_SIZE = 64
IMAGE_SIZE = 150
input_shape = (150,150,1) #reads a gray scale image of size 150x150

In [6]:
data_gen = ImageDataGenerator(rescale=1./255, validation_split = 0.2) # this is the way of reading imags from the directory using ImageDataGenerator 

In [7]:
train_gen = data_gen.flow_from_directory(image_dir,
                                         target_size = (IMAGE_SIZE,IMAGE_SIZE),
                                         batch_size = BATCH_SIZE,
                                         color_mode = 'grayscale',
                                         shuffle = True,
                                         class_mode = "binary",
                                         subset = "training"                                         
                                         )   # creating training sets of 80% of the image

Found 203 images belonging to 2 classes.


In [9]:
labels = train_gen.class_indices
classes = list(labels.keys())
print(classes) 

['no', 'yes']


In [10]:
val_gen = data_gen.flow_from_directory(image_dir,
                                         target_size = (IMAGE_SIZE,IMAGE_SIZE),
                                         batch_size = BATCH_SIZE,
                                         color_mode = 'grayscale',
                                         shuffle = False,
                                         class_mode = "binary",
                                         subset = "validation"                                         
                                         ) # creating validation images of 20%

Found 50 images belonging to 2 classes.


## Creating the model 

In [23]:
from keras.models import Sequential, load_model
from keras.layers import Dense,Conv2D,MaxPooling2D,BatchNormalization,Flatten,Dropout

In [ ]:
model = Sequential() # creating sequential model instances
model.add(keras.layers.InputLayer(input_shape = (150,150,1)))
model.add(Conv2D(16,(3,3), activation="relu"))
model.add(MaxPooling2D((2,2)))
model.add(Conv2D(32,(3,3), activation="relu"))
model.add(MaxPooling2D((2,2)))
model.add(Flatten())
model.add(Dense(512,activation="relu"))
model.add(Dropout(0.2))
model.add(Dense(1,activation='sigmoid'))

In [52]:
model.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics=['accuracy']) #compiling model using adam optimizer

In [53]:
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_7 (Conv2D)               │ (None, 148, 148, 16)   │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 74, 74, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 72, 72, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 36, 36, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 41472)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 512)            │    21,234,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │           513 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,239,489 (81.02 MB)

 Trainable params: 21,239,489 (81.02 MB)

 Non-trainable params: 0 (0.00 B)

In [54]:
history = model.fit(train_gen, verbose = 1, epochs=20, validation_data=val_gen) # training model with 20 epochs, tried with 10 epochs but accuracy was not good

Epoch 1/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 11s 3s/step - accuracy: 0.6415 - loss: 0.8830 - val_accuracy: 0.7400 - val_loss: 0.5203
Epoch 2/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 8s 2s/step - accuracy: 0.8008 - loss: 0.4931 - val_accuracy: 0.7600 - val_loss: 0.5220
Epoch 3/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 6s 1s/step - accuracy: 0.8160 - loss: 0.4616 - val_accuracy: 0.7800 - val_loss: 0.4875
Epoch 4/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 6s 2s/step - accuracy: 0.8668 - loss: 0.3531 - val_accuracy: 0.7400 - val_loss: 0.5123
Epoch 5/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 6s 1s/step - accuracy: 0.8971 - loss: 0.2663 - val_accuracy: 0.7800 - val_loss: 0.4858
Epoch 6/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 6s 1s/step - accuracy: 0.8977 - loss: 0.2699 - val_accuracy: 0.8400 - val_loss: 0.4545
Epoch 7/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 6s 1s/step - accuracy: 0.8975 - loss: 0.2326 - val_accuracy: 0.8000 - val_loss: 0.5256
Epoch 8/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 6s 1s/step - accuracy: 0.9299 - loss: 0.1923 - val_accuracy: 0.8600 - val_loss: 0.4402
Epoch 9/20
4/4 

# Evaluating the model

In [ ]:
test_loss, test_accuracy = model.evaluate(val_gen)
print(f" Accucary is {test_accuracy *100 :.2f}%") #wvaluating accuracy on test data or validation data

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 627ms/step - accuracy: 0.8600 - loss: 0.7750
 Accucary is 86.00%


In [56]:
model.save('brain_tumor_prediction_model_20_epochs.h5') # saving the model

# Prediction on New Image

In [24]:
from tensorflow.keras.preprocessing import image

loaded_model = load_model('brain_tumor_prediction_model.h5') # loading the saved model

In [ ]:
#image_path = 'D:/myvenv/archive/brain_tumor_dataset/no/N1.jpg'
image_path = 'D:/myvenv/archive/brain_tumor_dataset/yes/Y1.jpg'
img = image.load_img(image_path, target_size=(150,150), color_mode='grayscale') #reading image with size 150,150 and in grayscale format
img_array = image.img_to_array(img)/255.0 # normalizing the image from 0-255 to 0-1
img_array = np.expand_dims(img_array,axis=0) #this will attach batch information at the begining

prediction = loaded_model.predict(img_array)# perform prediction on loaded model
print("Tumor Detected" if prediction[0][0]>0.5 else "No Tumor detected") # print the result



1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
Tumor Detected


In [45]:
print("Tumor Detected" if prediction[0][0]>0.5 else "No Tumor detected")



Tumor Detected
